# MATH840 — Lab 9: Neural networks and foundation models

**Week 10 | graded lab | 8 points | due 23:59 today**

Your series from Week 4, for the last time. You have an ETS model (Lab 5), a seasonal ARIMA (Lab 7) and a
combination (Lab 8), each with a score on the hidden horizon. Tonight two new families challenge them:

- **neural networks** from the Week 9 lecture — MLP, N-BEATS, LSTM — trained on your series and nothing
  else;
- **foundation models** from the Week 10 lecture — Chronos-2, TimesFM 2.5, TiRex — which have never seen
  your series and are not trained on it at all.

Same four rolling windows, same benchmark, same MASE. Then you decide what to submit, and what ten weeks
of forecasting this series have taught you.

No plots of the series, no decomposition, no ACF: you have read your series four times.

**Accuracy is marked**, as in Labs 5, 7 and 8: beat your benchmark — 3 points, within 10% — 2, worse — 1
with a written explanation.

**Keep the section headings below.** They map one-to-one onto the marking rubric:

| Section | Criteria | Points |
|---|---|---|
| 1 | Your champion, rebuilt, and your past results | 1 |
| 2 | Forecast accuracy against your benchmark (scored from your file) | 3 |
| 3 | Justification block: neural networks, foundation models, conclusions against the whole course | 3 |
| 4 | Code quality and reproducibility | 1 |

Submit `SURNAME_lab10_forecast.csv`, `SURNAME_lab10.pdf` and `SURNAME_lab10.ipynb` to the Week 10
activity on Moodle before 23:59.

## 0. Setup

Given — run it and move on. Nothing in this section is marked. The first run downloads about 1.3 GB of
model weights; a CPU runtime is enough.

In [ ]:
!pip install -q statsforecast==2.1.1 neuralforecast==3.3.0 chronos-forecasting==2.3.2 "timesfm[torch]==3.0.2" tirex-ts==1.4.2

import hashlib
import logging
import os
import time
import warnings
warnings.filterwarnings("ignore")
os.environ["NIXTLA_ID_AS_COL"] = "true"
for name in ("lightning", "lightning.pytorch", "pytorch_lightning", "transformers"):
    logging.getLogger(name).setLevel(logging.CRITICAL)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from statsforecast.models import ARIMA, AutoARIMA, AutoETS, MSTL

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 120, "display.precision", 3)
print("running on", DEVICE)

In [ ]:
SURNAME = ""   # <-- for your submission file names
MY_ID   = ""   # <-- the SAME identifier you have used since Week 1

if not SURNAME.strip() or not MY_ID.strip():
    raise ValueError("Set SURNAME and MY_ID before running the rest.")

### Your series

The same identifier produces the same series as in Weeks 4, 6, 7, 8 and 9. If the code printed below is
not the one from your earlier submissions, stop and check `MY_ID`.

In [ ]:
BASE = ("https://raw.githubusercontent.com/Aranaur/aranaur.rbind.io/"
        "main/lectures/kse/MATH840/26autumn/data/trackb")


def assign_series(student_id: str, codes: list[str]) -> str:
    digest = hashlib.sha256(student_id.strip().encode("utf-8")).hexdigest()
    return codes[int(digest, 16) % len(codes)]


index = pd.read_csv(f"{BASE}/index.csv")
CODE = assign_series(MY_ID, sorted(index["code"]))
spec = index.set_index("code").loc[CODE]

M    = int(spec["m"])       # seasonal period: 12 monthly, 4 quarterly
H    = int(spec["h"])       # how many steps you are forecasting
FREQ = spec["freq"]         # pandas frequency of the calendar: "MS" or "QS"

s = pd.read_csv(f"{BASE}/{CODE}.csv", parse_dates=["ds"])
y = s["y"].to_numpy(float)
dates = s["ds"]

print(f"series {CODE}: {len(s)} observations, {dates.min().date()} to {dates.max().date()}")
print(f"seasonal period m = {M}, forecast horizon h = {H}, frequency {FREQ}")

### Helpers: the benchmark and your models from Labs 5, 7 and 8

Given, and the same as in Lab 8.

In [ ]:
def benchmark_forecasts(history: np.ndarray, h: int, m: int) -> dict[str, np.ndarray]:
    # The four simple methods from Week 1.
    T = len(history)
    return {
        "mean":   np.repeat(history.mean(), h),
        "naive":  np.repeat(history[-1], h),
        "snaive": np.array([history[-m + (i % m)] for i in range(h)]),
        "drift":  history[-1] + np.arange(1, h + 1) * (history[-1] - history[0]) / (T - 1),
    }


SCALE = float(np.mean(np.abs(y[M:] - y[:-M])))      # MASE denominator, fixed for this series


def mase(actual, forecast) -> float:
    return float(np.mean(np.abs(np.asarray(actual) - np.asarray(forecast))) / SCALE)


def rmse(actual, forecast) -> float:
    return float(np.sqrt(np.mean((np.asarray(actual) - np.asarray(forecast)) ** 2)))


train, valid = y[:-H], y[-H:]
simple = benchmark_forecasts(train, H, M)
BENCHMARK = min(simple, key=lambda name: rmse(valid, simple[name]))
print(f"your benchmark: {BENCHMARK} (validation MASE {mase(valid, simple[BENCHMARK]):.3f})")

FUTURE = pd.date_range(dates.iloc[-1], periods=H + 1, freq=FREQ)[1:]
print("you are forecasting:", FUTURE[0].date(), "to", FUTURE[-1].date())


class Fitted:
    # A fitted model whose forecasts are on the original scale, whatever it was fitted on.
    def __init__(self, model, log):
        self.model, self.log = model, log

    def predict(self, h: int, level=None):
        out = self.model.predict(h=h, level=level)
        return {k: np.exp(v) for k, v in out.items()} if self.log else out


def fit_ets(history, spec=None, damped=None, log=False) -> Fitted:
    # Lab 5: spec=None lets AutoETS search; "MAM", "AAdA" (damped) and so on fix the model.
    x = np.log(history) if log else np.asarray(history, float)
    if spec is None:
        return Fitted(AutoETS(season_length=M).fit(y=x), log)
    spec = str(spec).strip().upper()
    if spec[2:4] in ("DA", "DM", "DN"):
        spec, damped = spec[:2] + spec[3:], True
    return Fitted(AutoETS(season_length=M, model=spec,
                          damped=bool(damped) if spec[1] != "N" else None).fit(y=x), log)


def fit_arima(history, order=None, seasonal=(0, 0, 0), log=False, drift=False) -> Fitted:
    # Lab 7: order=(p, d, q) and seasonal=(P, D, Q) for a hand specification; order=None for AutoARIMA.
    x = np.log(history) if log else np.asarray(history, float)
    if order is None:
        model = AutoARIMA(season_length=M)
    else:
        model = ARIMA(order=tuple(order), seasonal_order=tuple(seasonal), season_length=M,
                      include_drift=drift and (order[1] + seasonal[1]) == 1)
    return Fitted(model.fit(y=x), log)


def fit_mstl(history, log=False) -> Fitted:
    # Lab 8: STL with the seasonal period M, an automatic ETS without a season for the rest.
    x = np.log(history) if log else np.asarray(history, float)
    return Fitted(MSTL(season_length=M).fit(y=x), log)


FITTERS = {"ets": fit_ets, "arima": fit_arima, "mstl": fit_mstl}


def fit_member(member, history) -> Fitted:
    kind, args = member
    return FITTERS[kind](history, **args)

### Helpers: the new families

A **forecaster** is a function `f(history, h)` that returns a dictionary with the point forecast `"mean"`
and the 80% bounds `"lo-80"`, `"hi-80"` (NaN when the method has no interval of its own). Three makers:

- `classic(member)` — a Lab 5/7/8 model;
- `neural(MLP)`, `neural(NBEATS)`, `neural(LSTM)` — a network trained **on `history` only**, from scratch,
  every time it is called;
- `foundation("Chronos-2")`, `foundation("TimesFM 2.5")`, `foundation("TiRex")` — zero-shot, the 10% and
  90% quantiles as the interval.

In [ ]:
from neuralforecast import NeuralForecast
from neuralforecast.models import LSTM, MLP, NBEATS
from chronos import BaseChronosPipeline
import timesfm
import tirex

chronos2 = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map=DEVICE)
tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
tfm.compile(timesfm.ForecastConfig(max_context=512, max_horizon=128, normalize_inputs=True,
                                   use_continuous_quantile_head=True, fix_quantile_crossing=True))
tirex_model = tirex.load_model("NX-AI/TiRex", device=DEVICE)
NAN = lambda h: np.full(h, np.nan)


def classic(member):
    def forecast(history, h):
        return fit_member(member, history).predict(h=h, level=[80])
    return forecast


def neural(model_class, input_size=None, max_steps=300, seed=1, **settings):
    # Trained on the history it is given and nothing else. input_size: how many past steps it reads.
    def forecast(history, h):
        df = pd.DataFrame({"unique_id": CODE, "ds": dates.iloc[:len(history)].to_numpy(), "y": history})
        model = model_class(h=h, input_size=input_size or 3 * M, max_steps=max_steps, scaler_type="robust",
                            random_seed=seed, accelerator="gpu" if DEVICE == "cuda" else "cpu", devices=1,
                            enable_progress_bar=False, enable_model_summary=False, logger=False, **settings)
        nf = NeuralForecast(models=[model], freq=FREQ)
        nf.fit(df=df)
        return {"mean": nf.predict()[model_class.__name__].to_numpy(), "lo-80": NAN(h), "hi-80": NAN(h)}
    return forecast


def foundation(name):
    def forecast(history, h):
        context = torch.tensor(np.asarray(history, float), dtype=torch.float32)
        if name == "Chronos-2":
            q, _ = chronos2.predict_quantiles([context], prediction_length=h, quantile_levels=[0.1, 0.5, 0.9])
            q = np.asarray(q[0]).reshape(-1, h, 3)[0]
            return {"mean": q[:, 1], "lo-80": q[:, 0], "hi-80": q[:, 2]}
        if name == "TimesFM 2.5":
            _, q = tfm.forecast(horizon=h, inputs=[context.numpy()])
            q = np.asarray(q)[0, :h]                       # mean, then the 10%, 20%, ..., 90% quantiles
            return {"mean": q[:, 5], "lo-80": q[:, 1], "hi-80": q[:, 9]}
        if name == "TiRex":
            q, _ = tirex_model.forecast(context=[context], prediction_length=h)
            q = np.asarray(q)[0]                           # the 10%, 20%, ..., 90% quantiles
            return {"mean": q[:, 4], "lo-80": q[:, 0], "hi-80": q[:, 8]}
        raise ValueError(name)
    return forecast

### Helpers: one protocol for everything

`backtest` runs every forecaster on the same four rolling origins as Lab 8 (window 1 is the validation
window of every lab since Week 4) and records how long each took. `combine`, `window_table`,
`conformal_radius` and `interval_check` are Lab 8's.

In [ ]:
WINDOWS = 4


def backtest(forecasters: dict, history: np.ndarray = y, h: int = H, windows: int = WINDOWS) -> pd.DataFrame:
    frames, seconds = [], {}
    for k in range(windows, 0, -1):
        end = len(history) - (k - 1) * h
        past, actual = history[:end - h], history[end - h:end]
        frame = pd.DataFrame({"window": k, "step": np.arange(1, h + 1), "actual": actual,
                              "benchmark": benchmark_forecasts(past, h, M)[BENCHMARK]})
        for name, f in forecasters.items():
            start = time.time()
            fc = f(past, h)
            seconds[name] = seconds.get(name, 0.0) + time.time() - start
            frame[name], frame[f"{name} lo"], frame[f"{name} hi"] = fc["mean"], fc["lo-80"], fc["hi-80"]
        frames.append(frame)
    bt = pd.concat(frames, ignore_index=True)
    bt.attrs["seconds"] = seconds
    return bt


def join(*backtests) -> pd.DataFrame:
    # Several backtests of the same windows side by side.
    out = backtests[0].copy()
    for other in backtests[1:]:
        new = [c for c in other.columns if c not in out.columns]
        out[new] = other[new].to_numpy()
        out.attrs["seconds"] = {**out.attrs.get("seconds", {}), **other.attrs.get("seconds", {})}
    return out


def combine(bt: pd.DataFrame, names: list[str], label: str) -> None:
    # Equal-weight average of the forecasts, and the average of their bounds (NaN if any has none).
    bt[label] = bt[names].mean(axis=1)
    bt[f"{label} lo"] = bt[[f"{n} lo" for n in names]].mean(axis=1, skipna=False)
    bt[f"{label} hi"] = bt[[f"{n} hi" for n in names]].mean(axis=1, skipna=False)


def window_table(bt: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    # MASE of each forecast in each window, the mean over windows, how often it was best, and its cost.
    table = bt.groupby("window").apply(
        lambda g: pd.Series({c: mase(g["actual"], g[c]) for c in columns}), include_groups=False).T
    table = table[sorted(table.columns, reverse=True)]
    table.columns = [f"window {w}" + (" (validation)" if w == 1 else "") for w in table.columns]
    table["mean"] = table.mean(axis=1)
    best = bt.groupby("window").apply(
        lambda g: min(columns, key=lambda c: mase(g["actual"], g[c])), include_groups=False)
    table["windows won"] = [int((best == c).sum()) for c in columns]
    table["seconds"] = [bt.attrs.get("seconds", {}).get(c, np.nan) for c in columns]
    return table


def winkler(actual, lo, hi, alpha=0.2) -> float:
    # Width of the interval plus 2/alpha times every miss (lecture, Week 8), in units of SCALE.
    actual, lo, hi = map(np.asarray, (actual, lo, hi))
    return float(np.mean((hi - lo) + 2 / alpha * (np.maximum(lo - actual, 0) + np.maximum(actual - hi, 0))) / SCALE)


def conformal_radius(errors) -> float:
    # The 80th percentile of the absolute errors: the forecast plus or minus this is the 80% interval.
    return float(np.quantile(np.abs(np.asarray(errors)), 0.80))


def interval_check(bt: pd.DataFrame, label: str) -> pd.DataFrame:
    # The forecaster's own interval (if it has one) and the conformal one, each judged on windows it did not
    # see: the conformal radius for a window comes from the errors of the OTHER windows only.
    rows = []
    for w, g in bt.groupby("window"):
        others = bt[bt["window"] != w]
        r = conformal_radius(others["actual"] - others[label])
        recipes = [("conformal", g[label] - r, g[label] + r)]
        if g[f"{label} lo"].notna().all():
            recipes.insert(0, ("own interval", g[f"{label} lo"], g[f"{label} hi"]))
        for recipe, lo, hi in recipes:
            rows.append({"recipe": recipe, "window": w,
                         "coverage": float(np.mean((g["actual"] >= lo) & (g["actual"] <= hi))),
                         "Winkler": winkler(g["actual"], lo, hi), "width": float(np.mean(hi - lo)) / SCALE})
    return pd.DataFrame(rows).groupby("recipe")[["coverage", "Winkler", "width"]].mean()

## 1. Your champion, rebuilt, and your past results

*1 point.* The **champion** is what you submitted in Lab 8: copy its members, the members you averaged,
and its interval recipe. If you did not submit Lab 8, your champion is the better of your Lab 5 and Lab 7
models, as a one-member combination — and say so below.

Then copy the scores you were sent after Labs 5, 7 and 8 into `PAST`.

In [ ]:
MEMBERS = {
    "ETS (Lab 5)":    ("ets", {"spec": None}),                     # <-- your Lab 5 submission
    "ARIMA (Lab 7)":  ("arima", {"order": None, "log": False}),     # <-- your Lab 7 submission
    "MSTL":           ("mstl", {"log": False}),                     # <-- as in Lab 8
}
CHAMPION = list(MEMBERS)          # <-- the members you averaged in Lab 8
CHAMPION_INTERVAL = "conformal"   # <-- "conformal" or "own interval" (Lab 8's averaged bounds)

PAST = {                          # <-- the scores you were sent: MASE and skill on the hidden horizon
    "Lab 5 (ETS)":            {"hidden MASE": None, "skill": None},
    "Lab 7 (seasonal ARIMA)": {"hidden MASE": None, "skill": None},
    "Lab 8 (combination)":    {"hidden MASE": None, "skill": None},
}

Run once — it refits your members on the four windows (as in Lab 8, a few minutes for a monthly ARIMA).
**The champion's validation-window MASE (window 1) must match your Lab 8 table.**

In [ ]:
bt_old = backtest({name: classic(member) for name, member in MEMBERS.items()})
combine(bt_old, CHAMPION, "champion")
window_table(bt_old, list(MEMBERS) + ["champion", "benchmark"])

**Does the champion match your Lab 8 numbers? If not, why?**

→

## 2. The challengers, and your forecast

*3 points, scored from your submission file* on the hidden horizon against your benchmark:
skill = your MASE / your benchmark's MASE; below 1 — 3 points, up to 1.1 — 2, above — 1.

What you may submit: **any one forecaster from your scoreboard (2.3), or an equal-weight average of two
or three of them** — old, neural or foundation, mixed as you like — with an 80% interval. Chosen on the
rolling windows, never on anything else.

⚠️ **Train only on your own series.** Several Track B series are cut from the same source; another
student's history can contain your hidden horizon. A network trained on the other series would be
leakage, and is not allowed.

### 2.1 Neural networks, trained on your series

The settings below are a reasonable start: the network reads the last `3 * M` observations and trains
for 300 steps. You may change `input_size`, `max_steps` or `seed` — say what you changed and why in 3.1.

In [ ]:
NEURAL = {
    "MLP":     neural(MLP),
    "N-BEATS": neural(NBEATS),
    "LSTM":    neural(LSTM),
}
bt_nn = backtest(NEURAL)
window_table(bt_nn, list(NEURAL) + ["benchmark"])

In [ ]:
# TODO (for 3.1): one more run of your best network with another seed, e.g. neural(NBEATS, seed=2),
# to see how much of its score is the network and how much is the luck of its starting weights.

### 2.2 Foundation models, zero-shot

No training: each model reads the history of the window and forecasts. Each returns its own 80% interval
(the 10% and 90% quantiles).

In [ ]:
FOUNDATION = {name: foundation(name) for name in ["Chronos-2", "TimesFM 2.5", "TiRex"]}
bt_fm = backtest(FOUNDATION)
window_table(bt_fm, list(FOUNDATION) + ["benchmark"])

### 2.3 The scoreboard

Everything on the same four windows: your champion and its members, the networks, the foundation models
and the benchmark.

**Before you read it**, write down what you expect, and which average of old and new you will add to the
table — one combination decided now, not after the table.

**What I expect, and the combination I add before reading the table.**

→

In [ ]:
bt = join(bt_old, bt_nn, bt_fm)
MY_MIX = ["champion", "Chronos-2"]           # <-- decided BEFORE the table: two or three forecasters
combine(bt, MY_MIX, "my mix")

everything = list(MEMBERS) + ["champion"] + list(NEURAL) + list(FOUNDATION) + ["my mix", "benchmark"]
scoreboard = window_table(bt, everything)
scoreboard.sort_values("mean")

In [ ]:
# TODO: one plot of your choice that compares the three families across the windows
# (for example: mean MASE as bars coloured by family, or MASE by window as lines).

### 2.4 Your choice and its interval

`CHOICE` is one column of the scoreboard, or an average you build with `combine` from two or three
columns. Then check both interval recipes on windows they did not see.

In [ ]:
CHOICE = "champion"                           # <-- a column of the scoreboard, e.g. "my mix" or "Chronos-2"
interval_check(bt, CHOICE)

### 2.5 The submission

Produce the forecast of your choice from the **whole** history `y`: every forecaster in it is called once
more with `y` (the networks train again). For the conformal interval the radius comes from **all four**
windows of `bt`.

In [ ]:
# TODO: your final forecast, from all of y.
#
#   ALL = {**{n: classic(m) for n, m in MEMBERS.items()}, **NEURAL, **FOUNDATION}
#   parts = ["Chronos-2"]                        # the forecasters inside CHOICE (for "champion": CHAMPION)
#   final = {n: ALL[n](y, H) for n in parts}
#   point = np.mean([final[n]["mean"] for n in parts], axis=0)
#   own interval: np.mean([final[n]["lo-80"] ...]), np.mean([final[n]["hi-80"] ...])
#   conformal:    r = conformal_radius(bt["actual"] - bt[CHOICE]); point - r, point + r
YHAT = None          # np.array of length H
YHAT_LO_80 = None    # np.array of length H
YHAT_HI_80 = None    # np.array of length H

for name, value in [("YHAT", YHAT), ("YHAT_LO_80", YHAT_LO_80), ("YHAT_HI_80", YHAT_HI_80)]:
    if value is None or len(np.ravel(value)) != H or np.isnan(np.ravel(value)).any():
        raise ValueError(f"{name} must be an array of length H = {H} with no NaN")

## 3. Justification block

*3 points, and the part a human reads closely.* Roughly a paragraph each. Numbers from your own cells,
not adjectives.

**3.1 Neural networks on your series** *(1)* — how MLP, N-BEATS and LSTM did against your champion and
the benchmark, window by window. Why: how many observations they learned from, what `input_size` means
for your series, what the second seed showed. What they cost in seconds. Anything you changed, and why.

→

**3.2 Foundation models** *(1)* — the same comparison for Chronos-2, TimesFM 2.5 and TiRex, and their
intervals against the conformal one. Your series is disguised public data: can you rule out that one of
them saw its source in pretraining? What would you need to know, and how does that change how far you
trust these numbers?

→

**3.3 What you submit, against the whole course** *(1)* — your Lab 5, 7 and 8 hidden-horizon skills
(`PAST`) next to tonight's rolling evidence. What you submitted and why; the skill you expect, and why.
If you had to forecast this series at work every month, which approach would you use and what would you
check — one paragraph.

→

In [ ]:
# Optional, for 3.3: your past results next to tonight's rolling means.
pd.DataFrame(PAST).T

## 4. Code quality and reproducibility

*1 point.* Before you export:

- Restart the kernel and run everything top to bottom. It must finish without errors.
- The submission file is written by the cell below, not edited by hand.
- No leakage: everything is compared on the rolling windows; only the final forecast sees all of `y`; no
  network is trained on any series but yours.
- Every number in your text comes from a cell.
- If you used TimeGPT (optional): the key comes from a Colab secret, never from the notebook.

## 5. Submit

Run both cells, then upload **three files** to the Week 10 activity on Moodle before 23:59:

- `SURNAME_lab10_forecast.csv` — written below;
- `SURNAME_lab10.pdf` — `File → Print → Save as PDF`;
- `SURNAME_lab10.ipynb` — `File → Download → Download .ipynb`.

In [ ]:
submission = pd.DataFrame({
    "student_id": MY_ID,
    "code": CODE,
    "ds": FUTURE.strftime("%Y-%m-%d"),
    "yhat": np.ravel(YHAT).astype(float),
    "yhat_lo_80": np.ravel(YHAT_LO_80).astype(float),
    "yhat_hi_80": np.ravel(YHAT_HI_80).astype(float),
})

name = f"{SURNAME.strip().upper()}_lab10_forecast.csv"
submission.to_csv(name, index=False)
print(submission.to_string(index=False))

In [ ]:
# The same checks used when your file is read. If this cell passes, your file is fine.
check = pd.read_csv(name, parse_dates=["ds"])
assert len(check) == H, f"expected {H} rows, got {len(check)}"
assert list(check.columns) == ["student_id", "code", "ds", "yhat", "yhat_lo_80", "yhat_hi_80"]
assert (check["ds"].to_numpy() == FUTURE.to_numpy()).all(), "the dates are not the H dates asked for"
assert check[["yhat", "yhat_lo_80", "yhat_hi_80"]].notna().all().all(), "no NaN allowed"
assert (check["yhat_lo_80"] <= check["yhat"]).all() and (check["yhat"] <= check["yhat_hi_80"]).all()
print("format OK -", name, "is ready to upload")

try:
    from google.colab import files
    files.download(name)
except ImportError:
    print(f"Not in Colab - {name} is saved next to this notebook.")